In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import polars as pl
from sometria.catalog import (
    MotionViewSpec,
    build_motion_view,
    load_annotations,
    load_catalog,
    load_label_vocabulary,
    load_splits,
)
from sometria.preprocess import ImportConfig, import_opensim_csv_dataset
from sometria.carepd import (
    UPDRS_COHORTS,
    import_carepd_annotations,
    import_carepd_folds,
    import_carepd_gait_vocabulary,
)
from sometria.representation import Representation, _load_human_definition

# CARE-PD

CARE-PD reaches us as the same OpenSim CSV as AMASS and MotionX, so the motion side is
the same import call. Three things differ:

- **The export is flat.** `<cohort>__<subject>__<take>.csv`, no directories, so the subset
  comes from the filename prefix. `_source_subset` in `sometria.preprocess` handles both.
- **It is an evaluation corpus, not pretraining data.** Nothing here calls
  `create_pretrain_split`: training on CARE-PD would train on the probe's own patients.
- **The splits are the release's, not ours.** CARE-PD ships participant lists per cohort;
  `import_carepd_folds` resolves them onto our takes.

Labels come from the released cohort pickles (`UPDRS_GAIT`), not from the torque export.
Four of the nine cohorts carry them -- 3DGait, BMCLab, PD-GaM, T-SDU-PD -- for 2,952 of
the 8,459 takes. The other five carry medication, freezer and disease-status labels, which
are different ontologies and are not imported here.

> Adeli et al., *CARE-PD: A Multi-Site Anonymized Clinical Dataset for Parkinson's Disease
> Gait Assessment*, NeurIPS 2025. Released CC BY-NC; cite the per-cohort papers too.

In [ ]:
RAW_CAREPD = Path("/home/fziche/nas/MAEVE/HUMAN_MODEL/CARE-PD_torque")
# the HuggingFace release: Canonicalized_SMPL_pickles/ for the labels, folds/ for the splits
HF_CAREPD = Path("/home/fziche/nas/MAEVE/HUMAN_MODEL/CARE_PD/hf")
OUT = Path("../data/processed")

HUMAN = _load_human_definition("../config/human.yaml")
REPRESENTATION = Representation.from_human(HUMAN)
print(REPRESENTATION)

## Import

`pattern="*.csv"` rather than `"**/*.csv"` on purpose. The export root also holds an older
nested copy of itself under `CARE-PD_torque/`, plus ~220 empty directories left by takes
whose conversion failed. A recursive glob would import the stale copy a second time under
different sample ids.

In [ ]:
catalog = import_opensim_csv_dataset(
    config=ImportConfig(
        source_dataset="CARE-PD",
        input_root=RAW_CAREPD,
        output_root=OUT,
        pattern="*.csv",
    ),
    representation=REPRESENTATION,
    human=HUMAN,
)

print(f"{len(catalog):,} catalog rows total")
catalog.head()

## Labels and splits

Three tables for the same reason BABEL uses three: a sample can be imported without being
labelled (5,507 of the takes are), and a benchmark's admissible answers are a different
question from what any one sample is.

In [ ]:
annotations = import_carepd_annotations(output_root=OUT, carepd_root=HF_CAREPD)
vocabulary = import_carepd_gait_vocabulary(output_root=OUT)
# An alternative vocabulary that drops score 3 from the data entirely. Not what the probe
# uses: the paper keeps all four classes and excludes 3 from the macro *average* only
# (S4.3), which config/experiment_probe_carepd.yaml does via `f1_label_subsets`. Kept
# imported so the stricter variant is reproducible, and because upsert_table never
# deletes -- a trimmed carepd_updrs_gait would leave its "3" row behind.
vocabulary_3class = import_carepd_gait_vocabulary(
    output_root=OUT, label_set="carepd_updrs_gait_3class", scores=(0, 1, 2)
)
splits = import_carepd_folds(output_root=OUT, carepd_root=HF_CAREPD)

print(f"{len(annotations):,} annotations | {len(vocabulary)} classes | "
      f"{splits.filter(pl.col('split_set').str.starts_with('carepd_'))['split_set'].n_unique()} split sets")


## Sanity checks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

catalog = load_catalog(OUT)
splits = load_splits(OUT)
annotations = load_annotations(OUT)

carepd = catalog.filter(pl.col("source_dataset") == "CARE-PD")
labels = annotations.filter(pl.col("label_source") == "CARE-PD")
labelled = carepd.join(labels.select("sample_id", "label"), on="sample_id", how="inner")

print(f"{len(carepd):,} samples across {carepd['source_subset'].n_unique()} cohorts")
print(f"{carepd['duration'].sum() / 3600:.1f} motion hours ({carepd['duration'].median():.1f}s median take)")
print(f"{len(labelled):,} labelled | {len(carepd) - len(labelled):,} unlabelled")
print(f"{carepd['broken'].sum():,} flagged broken")
carepd.head(3)

### Cohorts, take length, capture rate

CARE-PD takes are short -- seconds of walking, not a minutes-long mocap session -- and the
capture rate varies 25-150 Hz across cohorts. Both matter for window sizing: a probe window
has to be sized against this distribution, not against AMASS's.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

cohort_summary = (
    carepd
    .group_by("source_subset")
    .agg(
        pl.len().alias("samples"),
        (~pl.col("broken")).sum().alias("clean"),
        pl.col("duration").median().round(1).alias("median_s"),
        pl.col("original_hz").median().round(0).alias("capture_hz"),
        (100 * pl.col("broken").mean()).round(1).alias("broken_pct"),
    )
    .sort("samples")
)

y = np.arange(len(cohort_summary))
ax[0].barh(y - 0.2, cohort_summary["samples"], height=0.4, color="#999999", label="all")
ax[0].barh(y + 0.2, cohort_summary["clean"], height=0.4, color="#4878a8", label="clean")
ax[0].set_yticks(y, cohort_summary["source_subset"])
ax[0].set(title="Samples per cohort", xlabel="samples")
ax[0].legend()

ax[1].hist(carepd["duration"].to_numpy(), bins=80, color="#4878a8")
ax[1].axvline(240 / 60, color="#d1615d", linewidth=1.5, label="240 frames @ 60 Hz")
ax[1].set(title="Take duration", xlabel="seconds", ylabel="samples")
ax[1].legend()

ax[2].hist(carepd["original_hz"].to_numpy(), bins=80, color="#6acc64")
ax[2].set(title="Capture rate before resampling", xlabel="Hz", ylabel="samples")

fig.tight_layout()
print(cohort_summary.sort("samples", descending=True))

### The broken filter is not label-neutral here

This is the one result on this page that changes how the probe must be run.

`TAU_RATE_MAX` was calibrated on AMASS, as five robust sigmas above *its* median torque
rate. PD-GaM was captured at 25 Hz, and its median torque rate lands within a factor of two
of that threshold -- so the filter cuts that cohort near its own median, which is a coin
flip rather than a quality signal.

Because the cohorts are not balanced across severity, dropping them is not balanced either:
the surviving class distribution is materially different from the released one. A probe run
with `exclude_broken=True` is therefore not scoring CARE-PD's benchmark, and its numbers are
not comparable to the paper's. Decide the threshold before reporting anything.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

from sometria.preprocess import TAU_RATE_MAX

for cohort, group in sorted(carepd.group_by("source_subset"), key=lambda kv: kv[0][0]):
    ax[0].hist(
        np.log10(group["tau_rate"].to_numpy().clip(1)), bins=50, alpha=0.55, label=cohort[0]
    )
ax[0].axvline(np.log10(TAU_RATE_MAX), color="black", linewidth=1.5, label="TAU_RATE_MAX")
ax[0].set(xlabel="log10 max |dtau/dt|", ylabel="samples", title="Torque discontinuity by cohort")
ax[0].legend(fontsize=8)

by_class = (
    labelled
    .group_by("label")
    .agg(
        pl.len().alias("released"),
        (~pl.col("broken")).sum().alias("clean"),
        (100 * pl.col("broken").mean()).round(1).alias("broken_pct"),
    )
    .sort("label")
)

x = np.arange(len(by_class))
ax[1].bar(x - 0.2, by_class["released"], width=0.4, color="#999999", label="released")
ax[1].bar(x + 0.2, by_class["clean"], width=0.4, color="#4878a8", label="clean")
ax[1].set_xticks(x, by_class["label"])
ax[1].set(title="Class counts before and after the filter", xlabel="UPDRS gait score", ylabel="takes")
ax[1].legend()

ax[2].bar(x, by_class["broken_pct"], color="#d1615d")
ax[2].set_xticks(x, by_class["label"])
ax[2].set(title="Broken rate by class", xlabel="UPDRS gait score", ylabel="% dropped")

fig.tight_layout()
print(by_class)

### Official evaluation protocols

`import_carepd_folds` materializes every protocol the paper evaluates (S4.2) as a named
split set, so switching from within-site to cross-site is one config override:

| split set | train / eval | count |
|---|---|---|
| `carepd_<cohort>_loso_<k>` | leave-one-subject-out, the paper's headline | 110 |
| `carepd_<cohort>_6fold_<k>` | 6-fold participant CV | 24 |
| `carepd_<cohort>_fixed` | the release's fixed split | 4 |
| `carepd_cross_<x>_to_<y>` | train on x, test on y | 12 |
| `carepd_lodo_<x>` | train on the other three, test on x | 4 |
| `carepd_mida_<x>_loso_<k>` | LODO plus x's own LOSO train split | 110 |
| `carepd_fixed`, `carepd_6fold_<k>` | **ours**: the four cohorts pooled | 7 |

Only the first three come from released participant lists; the rest are cohort algebra
over them. The pooled sets are not the paper's -- every cohort appears on both sides, and
cohort priors differ enough (T-SDU-PD is 44% score-2 against PD-GaM's 15%) that a probe
can score by recognizing the capture site. Do not table those against published numbers.

The release ships participant lists, so every take of a patient lands on one side. Both
assertions below are the reason to use the release's folds rather than a split of our own.

In [ ]:
carepd_splits = splits.filter(pl.col("split_set").str.starts_with("carepd_"))

resolved = (
    carepd_splits
    .join(carepd.select("sample_id", "source_path", "source_subset"), on="sample_id", how="inner")
    .join(labels.select("sample_id", "label"), on="sample_id", how="inner")
    .with_columns(
        pl.concat_str(
            pl.col("source_subset"),
            pl.col("source_path").str.split("__").list.get(1),
            separator="/",
        ).alias("participant")
    )
)

for split_set in sorted(resolved["split_set"].unique()):
    one = resolved.filter(pl.col("split_set") == split_set)
    spans = one.group_by("participant").agg(pl.col("split").n_unique().alias("n"))
    assert spans["n"].max() == 1, f"{split_set} splits a participant across sides"
print(f"OK: participant-disjoint in all {resolved['split_set'].n_unique()} split sets")

held_out = (
    resolved
    .filter(pl.col("split_set").str.starts_with("carepd_6fold_") & (pl.col("split") == "eval"))
    .group_by("sample_id").len()
)
assert set(held_out["len"]) == {1}, "a take is held out in more than one fold"
print(f"OK: each of {len(held_out):,} takes is held out exactly once across the 6 folds")

### Held-out class coverage per fold

Score 3 is 45 takes from a handful of participants, and the release's own folds cannot
spread them evenly: some folds hold out one or none. A four-way head evaluated per fold
will score a class it never saw, which is why `config/experiment_probe_carepd.yaml`
uses the three-class vocabulary imported above. Dropping the label from the vocabulary
is not enough on its own: those takes would survive as windows with an all-zero target,
so the probe config also sets `require_in_vocabulary: true` to drop them outright.

In [ ]:
coverage = (
    resolved
    .filter(pl.col("split") == "eval")
    .group_by("split_set", "label").len()
    .pivot(on="label", index="split_set", values="len")
    .fill_null(0)
    .sort("split_set")
)
print(coverage)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))

classes = sorted(c for c in coverage.columns if c != "split_set")
bottom = np.zeros(len(coverage))
for label in classes:
    ax[0].bar(coverage["split_set"], coverage[label], bottom=bottom, label=f"score {label}")
    bottom += coverage[label].to_numpy()
ax[0].set_xticks(range(len(coverage)), coverage["split_set"], rotation=30, ha="right")
ax[0].set(title="Held-out takes per fold", ylabel="takes")
ax[0].legend(fontsize=8)

rare = coverage["3"].to_numpy() if "3" in coverage.columns else np.zeros(len(coverage))
ax[1].bar(coverage["split_set"], rare, color="#d1615d")
ax[1].set_xticks(range(len(coverage)), coverage["split_set"], rotation=30, ha="right")
ax[1].set(title="Held-out score-3 takes per fold", ylabel="takes")

fig.tight_layout()

### CARE-PD stays out of pretraining

`create_pretrain_split` defaults to "every corpus in the catalog", which was correct while
every corpus was pretraining data. Re-running the AMASS or MotionX notebook must now pass
`source_datasets=("AMASS", "MotionX")`, or the probe's own patients land in the pretraining
view. This cell fails loudly if that has happened.

In [ ]:
pretrain_ids = splits.filter(pl.col("split_set").str.starts_with("pretrain"))
leaked = carepd.join(pretrain_ids.select("sample_id", "split_set").unique(), on="sample_id", how="inner")
assert leaked.is_empty(), (
    f"{len(leaked)} CARE-PD samples are in {leaked['split_set'].unique().to_list()}; "
    "rebuild those splits with source_datasets=(\"AMASS\", \"MotionX\")"
)
print("OK: no CARE-PD sample is in any pretraining split")

### What a probe view sees

In [ ]:
rows = []
for split_set in ("carepd_fixed", "carepd_6fold_1"):
    for split in ("train", "eval"):
        for exclude_broken in (False, True):
            view = build_motion_view(
                OUT,
                MotionViewSpec(
                    split_set=split_set,
                    split=split,
                    source_datasets=("CARE-PD",),
                    label_sources=("CARE-PD",),
                    require_labels=True,
                    exclude_broken=exclude_broken,
                ),
            )
            rows.append({
                "split_set": split_set,
                "split": split,
                "exclude_broken": exclude_broken,
                "samples": len(view),
                "hours": round(view["duration"].sum() / 3600, 2),
                "min_frames": view["n_frames"].min(),
            })

print(pl.DataFrame(rows))
print(load_label_vocabulary(OUT, "carepd_updrs_gait"))